# scipy — runtime smoke test

In [ ]:
# Example 1 — scipy.special + scipy.stats: special-function identities and distribution
# cdf/ppf, t-test and KS checks, each against a known closed-form answer.
import numpy as np
from scipy import special, stats

assert np.isclose(special.gammaln(5.0), np.log(24.0)), special.gammaln(5.0)      # Gamma(5) = 4! = 24
assert np.isclose(special.erf(0.0), 0.0) and np.isclose(special.erf(np.inf), 1.0)
assert np.isclose(special.erfinv(special.erf(0.7)), 0.7)                          # exact inverse pair
assert np.isclose(special.betainc(2.0, 3.0, 1.0), 1.0)                            # regularized -> 1
assert np.isclose(special.expit(special.logit(0.3)), 0.3)
assert np.isclose(special.comb(10, 3, exact=False), 120.0), special.comb(10, 3)

n = stats.norm()
assert np.isclose(n.cdf(0.0), 0.5) and np.isclose(n.ppf(0.975), 1.959963984540054)
assert np.isclose(n.ppf(n.cdf(1.3)), 1.3)                                         # cdf/ppf round trip

rng = np.random.default_rng(0)
same_a, same_b = rng.normal(0.0, 1.0, 400), rng.normal(0.0, 1.0, 400)
shifted = rng.normal(3.0, 1.0, 400)
assert stats.ttest_ind(same_a, same_b).pvalue > 0.05, "t-test flagged two identical populations"
assert stats.ttest_ind(same_a, shifted).pvalue < 1e-10, "t-test missed a 3-sigma shift"
assert stats.ks_1samp(same_a, stats.norm.cdf).pvalue > 0.05
print("scipy.special identities + stats cdf/ppf/t-test/KS agree with closed form")

In [ ]:
# Example 2 — scipy.sparse: sparse-sparse matmul + matvec vs a dense reference.
import numpy as np
from scipy import sparse

rng = np.random.default_rng(0)
A = sparse.random_array((400, 400), density=0.02, format="csr", rng=rng)
B = sparse.random_array((400, 400), density=0.02, format="csr", rng=rng)

C = A @ B                                    # sparse @ sparse
assert sparse.issparse(C) and C.format == "csr"
max_err = np.abs(C.toarray() - A.toarray() @ B.toarray()).max()
print(f"nnz(A)={A.nnz}  nnz(A@B)={C.nnz}  max|sparse - dense|={max_err:.2e}")
assert max_err < 1e-9

x = rng.standard_normal(400)                 # sparse matvec
assert np.allclose(A @ x, A.toarray() @ x, atol=1e-9)
print("sparse matmul + matvec match dense — OK")

In [ ]:
# Example 3 — scipy.integrate.solve_ivp: integrate a damped harmonic oscillator
# y'' + 2*z*w*y' + w^2*y = 0 and check the numeric solution against the closed-form
# analytic solution on a dense grid. Exercises the RK45 integrator + dense output.
import numpy as np
from scipy.integrate import solve_ivp

w, z = 2.0, 0.1                              # natural freq, damping ratio (<1)
def rhs(t, y):
    return [y[1], -2*z*w*y[1] - w*w*y[0]]

sol = solve_ivp(rhs, (0.0, 10.0), [1.0, 0.0], method="RK45",
                dense_output=True, rtol=1e-8, atol=1e-10)
assert sol.success

wd = w*np.sqrt(1 - z*z)                       # damped frequency
def analytic(t):
    return np.exp(-z*w*t)*(np.cos(wd*t) + (z*w/wd)*np.sin(wd*t))

ts = np.linspace(0, 10, 200)
max_err = np.abs(sol.sol(ts)[0] - analytic(ts)).max()
print(f"steps={sol.t.size}  max|numeric - analytic|={max_err:.2e}")
assert max_err < 1e-5
print("solve_ivp matches analytic solution — OK")

In [ ]:
# Example 4 — scipy.optimize.minimize: unconstrained BFGS on the 5-D Rosenbrock
# (with analytic gradient) plus an equality-constrained SLSQP problem. Exercises
# two distinct optimizer backends and checks convergence to the known optima.
import numpy as np
from scipy.optimize import minimize, rosen, rosen_der

res = minimize(rosen, np.array([1.3, 0.7, 0.8, 1.9, 1.2]), method="BFGS",
               jac=rosen_der, options={"gtol": 1e-8})
print(f"BFGS: success={res.success} nit={res.nit} fun={res.fun:.2e}")
assert res.success and res.fun < 1e-8
assert np.allclose(res.x, 1.0, atol=1e-4)    # global min of Rosenbrock is all-ones

# minimize x^2 + y^2 subject to x + y = 1  ->  (0.5, 0.5), fun = 0.5
con = {"type": "eq", "fun": lambda v: v[0] + v[1] - 1.0}
r2 = minimize(lambda v: v @ v, [2.0, -3.0], method="SLSQP", constraints=[con])
print(f"SLSQP: x={r2.x}  fun={r2.fun:.4f}")
assert r2.success and np.allclose(r2.x, [0.5, 0.5], atol=1e-3)
assert abs(r2.x.sum() - 1.0) < 1e-6          # equality constraint satisfied
print("BFGS + SLSQP reached known optima — OK")

In [ ]:
# Example 5 — scipy.signal: design a 6th-order Butterworth low-pass (SOS form) and
# apply zero-phase filtering to a 5 Hz + 120 Hz mixture. Checks that the 120 Hz
# component is rejected (residual-vs-clean shrinks) and the freq response matches.
import numpy as np
from scipy import signal

fs = 1000.0
t = np.arange(0, 1.0, 1/fs)
clean = np.sin(2*np.pi*5*t)                   # 5 Hz — keep
x = clean + 0.5*np.sin(2*np.pi*120*t)         # 120 Hz — reject

sos = signal.butter(6, 30.0, btype="low", fs=fs, output="sos")
y = signal.sosfiltfilt(sos, x)                # zero-phase (forward-backward)

before = np.sqrt(np.mean((x - clean)**2))
after = np.sqrt(np.mean((y - clean)**2))
print(f"rms(residual) before={before:.4f} after={after:.4f}")
assert after < before/5                        # filtering removed most of the 120 Hz

_, h = signal.sosfreqz(sos, worN=[5.0, 120.0], fs=fs)
mag = np.abs(h)
print(f"|H(5Hz)|={mag[0]:.3f}  |H(120Hz)|={mag[1]:.5f}")
assert mag[0] > 0.9 and mag[1] < 0.05          # passband kept, stopband attenuated
print("Butterworth low-pass rejects 120 Hz — OK")

In [ ]:
# Example 6 — scipy.linalg vs numpy invariants: SVD, symmetric eig, dense solve and LU
# on a random matrix, each checked by reconstruction or against numpy.
import numpy as np
from scipy import linalg

rng = np.random.default_rng(1)
M = rng.standard_normal((120, 120))

U, s, Vt = linalg.svd(M)                       # SVD reconstruction + orthogonality
assert np.allclose((U * s) @ Vt, M, atol=1e-9)
assert np.allclose(U @ U.T, np.eye(120), atol=1e-9)

S = M + M.T                                    # symmetric eigendecomposition
vals, vecs = linalg.eigh(S)
assert np.allclose(vecs @ np.diag(vals) @ vecs.T, S, atol=1e-8)
assert np.all(np.diff(vals) >= -1e-12)         # eigh returns ascending eigenvalues

b = rng.standard_normal(120)                   # dense solve agrees with numpy
xs = linalg.solve(M, b)
assert np.allclose(M @ xs, b, atol=1e-8)
assert np.allclose(xs, np.linalg.solve(M, b), atol=1e-8)

P, L, Uu = linalg.lu(M)                         # LU round-trip
assert np.allclose(P @ L @ Uu, M, atol=1e-9)
print("SVD / eigh / solve / LU all satisfy invariants vs numpy — OK")

In [ ]:
# Example 7 — scipy.io.mmwrite / mmread: Matrix Market round-trip for a dense array and a
# sparse (coordinate) matrix, plus out-of-range values that read back as +/-inf.
import io, math
import numpy as np
import scipy.io
from scipy import sparse

dense = np.array([[1.5, -2.25, 0.0], [0.003, 4.0, 1.0e10]])
buf = io.BytesIO()
scipy.io.mmwrite(buf, dense)
assert buf.getvalue().startswith(b"%%MatrixMarket matrix array real general"), buf.getvalue()[:60]
back = scipy.io.mmread(io.BytesIO(buf.getvalue()))
assert isinstance(back, np.ndarray) and np.array_equal(back, dense), back

rng = np.random.default_rng(2)
coo = sparse.random_array((50, 40), density=0.05, format="coo", rng=rng)
buf = io.BytesIO()
scipy.io.mmwrite(buf, coo)
assert buf.getvalue().startswith(b"%%MatrixMarket matrix coordinate real general"), buf.getvalue()[:60]
back = scipy.io.mmread(io.BytesIO(buf.getvalue()), spmatrix=False)
assert sparse.issparse(back) and back.shape == coo.shape and back.nnz == coo.nnz
assert np.allclose(back.toarray(), coo.toarray(), rtol=0, atol=1e-15), "sparse round-trip changed values"

text = b"%%MatrixMarket matrix array real general\n3 1\n1e400\n-1e400\n2.5\n"
over = np.asarray(scipy.io.mmread(io.BytesIO(text))).reshape(-1)
assert over[0] == math.inf and over[1] == -math.inf and over[2] == 2.5, over
print(f"dense {dense.shape} + coordinate nnz={coo.nnz} round-trips; 1e400 -> {over[0]}, -1e400 -> {over[1]}")
print("scipy.io.mmwrite/mmread round-trip and overflow to inf: OK")